# Training-Time Ensemble (Train-Split-Only) Meta-Learners — 16 Features with Entropy (Fixed)

This notebook is the leakage-safe train-split-only stacking pipeline, extended with uncertainty features and fixed BiLSTM grouping.

## Feature design (16 total)
- 12 soft probabilities (`3 classes x 4 models`)
- 4 entropy values (`1 per model`)

Entropy adds uncertainty awareness for each model and helps meta-learners decide when to trust or down-weight a model.

## Leakage policy
- Meta-training data: **only `train` split** of each dataset config.
- Evaluation data: strictly held-out splits (`test` if available, else `validation_matched`/`validation_mismatched`).

## Meta-learners
- `MLP` (`16 -> 64 -> 32 -> 3`, ReLU + BatchNorm + Dropout 0.2, AdamW)
- `LinearSVC` (fast linear baseline for large train-split features)
- `BiLSTM` (grouped sequence: 4 model timesteps x 4 features each)


In [ ]:
import gc
import random
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

from datasets import Dataset, load_dataset
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    DataCollatorWithPadding,
)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

MODEL_IDS = {
    "bert": "emrecan/bert-base-turkish-cased-allnli_tr",
    "mdeberta": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "gemma": "google/gemma-3-27b-it",
    "qwen": "Qwen/Qwen2-7B-Instruct",
}

MODEL_ORDER = ["bert", "mdeberta", "gemma", "qwen"]
DATASET_NAME = "yilmazzey/sdp2-nli"
CONFIGS = ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]
LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}
LABEL_NAMES = [LABEL_MAP[i] for i in range(3)]
NUM_LABELS = 3

EVAL_SPLITS = {
    "snli_tr_1_1": ["test"],
    "multinli_tr_1_1": ["validation_matched", "validation_mismatched"],
    "trglue_mnli": ["test_matched", "test_mismatched"],
}

STATIC_WEIGHTED_WEIGHTS = {
    "bert": 0.25,
    "mdeberta": 0.12,
    "gemma": 0.26,
    "qwen": 0.37,
}

# Router first selects a provisional class mostly from Gemma/Qwen,
# then class-conditional model weights produce the final prediction.
ROUTER_W = {"bert": 0.0, "mdeberta": 0.0, "gemma": 0.5, "qwen": 0.5}
CLASS_WEIGHTS_V1 = {
    0: {"bert": 0.0, "mdeberta": 0.05, "gemma": 0.25, "qwen": 0.70},
    1: {"bert": 0.0, "mdeberta": 0.10, "gemma": 0.70, "qwen": 0.20},
    2: {"bert": 0.0, "mdeberta": 0.05, "gemma": 0.25, "qwen": 0.70},
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CACHE_DIR = Path("stacking_cache_train_split_16feat_entropy")
CACHE_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR = Path("stacking_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

ENCODER_BATCH_SIZE = 32
QWEN_BATCH_SIZE = 16
GEMMA_BATCH_SIZE = 8

META_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 5
META_BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

print("Cache directory:", CACHE_DIR.resolve())


In [ ]:
def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def load_encoder_models():
    print("Loading BERT:", MODEL_IDS["bert"])
    bert_tok = AutoTokenizer.from_pretrained(MODEL_IDS["bert"])
    bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_IDS["bert"]).to(DEVICE)
    bert_model.eval()
    print("  params:", f"{count_params(bert_model):,}")

    print("Loading mDeBERTa:", MODEL_IDS["mdeberta"])
    mdeb_tok = AutoTokenizer.from_pretrained(MODEL_IDS["mdeberta"])
    mdeb_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_IDS["mdeberta"], ignore_mismatched_sizes=True
    ).to(DEVICE)
    mdeb_model.eval()
    print("  params:", f"{count_params(mdeb_model):,}")

    return {
        "bert": (bert_model, bert_tok),
        "mdeberta": (mdeb_model, mdeb_tok),
    }


def nli_prompt_gemma(premise, hypothesis):
    return [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for natural language inference. "
                "Classify the relationship between premise and hypothesis as entailment, neutral, or contradiction. "
                "Respond with exactly one word: entailment, neutral, or contradiction. No explanation or additional text."
            ),
        },
        {"role": "user", "content": f"Premise: {premise}\\nHypothesis: {hypothesis}\\nLabel:"},
    ]


def nli_prompt_qwen(premise, hypothesis):
    return [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for natural language inference. "
                "Classify the relationship between premise and hypothesis as entailment, neutral, or contradiction. "
                "Respond with exactly one word only: entailment, neutral, or contradiction. No explanation, no other text."
            ),
        },
        {"role": "user", "content": f"Premise: {premise}\\nHypothesis: {hypothesis}\\nLabel:"},
    ]


def load_causal(model_id: str):
    use_bf16 = torch.cuda.is_available() or (
        getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
    )
    dtype = torch.bfloat16 if use_bf16 else torch.float32

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=dtype,
            device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=dtype,
        ).to(DEVICE)

    model.eval()
    return model, tok


def unload_causal(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def verbalizer_first_token_ids(tokenizer, word: str):
    variants = [word, " " + word, "\\n" + word, word + "\\n"]
    seen = []
    for v in variants:
        ids = tokenizer.encode(v, add_special_tokens=False)
        if ids:
            seen.append(ids[0])
    if not seen:
        return tokenizer.unk_token_id
    return seen[0]


def build_label_token_ids(tokenizer):
    tids = [verbalizer_first_token_ids(tokenizer, w) for w in LABEL_NAMES]
    if len(set(tids)) < NUM_LABELS:
        tids = []
        for w in LABEL_NAMES:
            ids = tokenizer.encode(" " + w, add_special_tokens=False)
            tids.append(ids[0] if ids else tokenizer.unk_token_id)
    return tids


encoders = load_encoder_models()
print("Encoder models ready on", DEVICE)


In [ ]:
def ensure_cached(path: Path, fn):
    if path.exists():
        return np.load(path)
    arr = fn()
    np.save(path, arr)
    return arr


@torch.no_grad()
def encoder_softmax_probs(model, tokenizer, ds: Dataset, batch_size: int, desc: str) -> np.ndarray:
    def tokenize_fn(examples):
        return tokenizer(
            examples["premise"],
            examples["hypothesis"],
            truncation=True,
            max_length=256,
        )

    remove_cols = [c for c in ds.column_names if c != "label"]
    tok_ds = ds.map(tokenize_fn, batched=True, remove_columns=remove_cols, desc=f"Tokenize {desc}")
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def collate_fn(examples):
        labels = torch.tensor([ex["label"] for ex in examples])
        batch = collator([{k: v for k, v in ex.items() if k != "label"} for ex in examples])
        batch["_labels"] = labels
        return batch

    loader = DataLoader(tok_ds, batch_size=batch_size, collate_fn=collate_fn)
    chunks = []
    model.eval()

    for batch in tqdm(loader, desc=desc):
        logits = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
        ).logits
        chunks.append(F.softmax(logits, dim=-1).cpu().numpy())

    return np.concatenate(chunks, axis=0)


@torch.no_grad()
def llm_soft_probs(model, tokenizer, ds: Dataset, prompt_fn, batch_size: int, desc: str) -> np.ndarray:
    premises = list(ds["premise"])
    hypotheses = list(ds["hypothesis"])
    n = len(ds)
    dev = next(model.parameters()).device
    label_ids = torch.tensor(build_label_token_ids(tokenizer), dtype=torch.long, device=dev)
    chunks = []

    for start in tqdm(range(0, n, batch_size), desc=desc):
        bp = premises[start : start + batch_size]
        bh = hypotheses[start : start + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                prompt_fn(p, h),
                tokenize=False,
                add_generation_prompt=True,
            )
            for p, h in zip(bp, bh)
        ]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
        enc = {k: v.to(dev) for k, v in enc.items()}

        logits = model(**enc).logits
        last_idx = enc["attention_mask"].sum(dim=1) - 1
        last_logits = logits[torch.arange(logits.size(0), device=dev), last_idx]
        selected = last_logits[:, label_ids]
        probs = F.softmax(selected.float(), dim=-1)
        chunks.append(probs.cpu().numpy())

    return np.concatenate(chunks, axis=0)


def cache_name(config: str, split: str, model_key: str) -> Path:
    safe_cfg = config.replace("/", "_")
    safe_sp = split.replace("/", "_")
    return CACHE_DIR / f"{safe_cfg}__{safe_sp}__{model_key}_probs.npy"


def ensure_cached_split_probs(config: str, split: str, ds_split: Dataset):
    probs = {}

    probs["bert"] = ensure_cached(
        cache_name(config, split, "bert"),
        lambda: encoder_softmax_probs(
            encoders["bert"][0], encoders["bert"][1], ds_split, ENCODER_BATCH_SIZE, f"{config}/{split} BERT"
        ),
    )
    probs["mdeberta"] = ensure_cached(
        cache_name(config, split, "mdeberta"),
        lambda: encoder_softmax_probs(
            encoders["mdeberta"][0], encoders["mdeberta"][1], ds_split, ENCODER_BATCH_SIZE, f"{config}/{split} mDeBERTa"
        ),
    )

    def compute_qwen():
        q_model, q_tok = load_causal(MODEL_IDS["qwen"])
        q_probs = llm_soft_probs(q_model, q_tok, ds_split, nli_prompt_qwen, QWEN_BATCH_SIZE, f"{config}/{split} Qwen")
        unload_causal(q_model)
        return q_probs

    def compute_gemma():
        g_model, g_tok = load_causal(MODEL_IDS["gemma"])
        g_probs = llm_soft_probs(g_model, g_tok, ds_split, nli_prompt_gemma, GEMMA_BATCH_SIZE, f"{config}/{split} Gemma")
        unload_causal(g_model)
        return g_probs

    probs["qwen"] = ensure_cached(cache_name(config, split, "qwen"), compute_qwen)
    probs["gemma"] = ensure_cached(cache_name(config, split, "gemma"), compute_gemma)

    return probs


In [ ]:
def compute_entropy(probs: np.ndarray) -> np.ndarray:
    """Entropy of a batch of probability distributions. Shape (N, 3) -> (N,)"""
    probs = np.clip(probs, 1e-10, 1.0)
    return -np.sum(probs * np.log(probs), axis=1)


def build_X16(probs_dict: dict) -> np.ndarray:
    """16 features: 12 probs + 4 per-model entropy values."""
    bert_p = probs_dict["bert"]
    mdeb_p = probs_dict["mdeberta"]
    gemma_p = probs_dict["gemma"]
    qwen_p = probs_dict["qwen"]

    ent_bert = compute_entropy(bert_p).reshape(-1, 1)
    ent_mdeb = compute_entropy(mdeb_p).reshape(-1, 1)
    ent_gemma = compute_entropy(gemma_p).reshape(-1, 1)
    ent_qwen = compute_entropy(qwen_p).reshape(-1, 1)

    return np.hstack([
        bert_p,
        mdeb_p,
        gemma_p,
        qwen_p,
        ent_bert,
        ent_mdeb,
        ent_gemma,
        ent_qwen,
    ]).astype(np.float32)


def onehot_from_probs(p: np.ndarray) -> np.ndarray:
    out = np.zeros_like(p)
    idx = p.argmax(axis=1)
    out[np.arange(len(idx)), idx] = 1.0
    return out


def weighted_static_pred(bert_p, mdeb_p, gemma_p, qwen_p):
    w = np.array([STATIC_WEIGHTED_WEIGHTS[m] for m in MODEL_ORDER], dtype=np.float32)
    probs = np.stack([bert_p, mdeb_p, gemma_p, qwen_p], axis=0)
    scores = np.einsum("m,mnc->nc", w, probs)
    return scores.argmax(axis=1)


def majority_vote_pred(bert_p, mdeb_p, gemma_p, qwen_p):
    votes = np.stack([bert_p.argmax(1), mdeb_p.argmax(1), gemma_p.argmax(1), qwen_p.argmax(1)], axis=1)
    out = []
    for row in votes:
        out.append(int(Counter(row.tolist()).most_common(1)[0][0]))
    return np.array(out, dtype=np.int64)


def class_conditional_routing_pred(bert_p, mdeb_p, gemma_p, qwen_p):
    oh = np.stack([
        onehot_from_probs(bert_p),
        onehot_from_probs(mdeb_p),
        onehot_from_probs(gemma_p),
        onehot_from_probs(qwen_p),
    ], axis=0)

    router_w = np.array([ROUTER_W[m] for m in MODEL_ORDER], dtype=np.float32)
    router_scores = np.einsum("m,mnc->nc", router_w, oh)
    routed_class = router_scores.argmax(axis=1)

    final = np.empty(len(routed_class), dtype=np.int64)
    for c in range(NUM_LABELS):
        mask = routed_class == c
        if not mask.any():
            continue
        w_c = np.array([CLASS_WEIGHTS_V1[c][m] for m in MODEL_ORDER], dtype=np.float32)
        s_c = np.einsum("m,mnc->nc", w_c, oh[:, mask, :])
        final[mask] = s_c.argmax(axis=1)
    return final


FEATURE_NAMES = [
    "bert_p0", "bert_p1", "bert_p2",
    "mdeberta_p0", "mdeberta_p1", "mdeberta_p2",
    "gemma_p0", "gemma_p1", "gemma_p2",
    "qwen_p0", "qwen_p1", "qwen_p2",
    "entropy_bert", "entropy_mdeberta", "entropy_gemma", "entropy_qwen",
]


In [ ]:
class StackingMLP(nn.Module):
    """16 -> 64 -> 32 -> 3, requested MLP architecture."""

    def __init__(self, in_dim=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class StackingBiLSTM(nn.Module):
    """Properly grouped BiLSTM: 4 timesteps (one per model) x 4 features each."""

    def __init__(self, hidden_size=64, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=4,          # 3 probs + 1 entropy per model
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,    # forward + backward across models
        )
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size * 2, n_classes)

    def forward(self, x):
        # x shape: (batch, 16) -> (batch, 4 timesteps, 4 features)
        x = x.view(x.size(0), 4, 4)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(self.drop(last))


def _train_torch_model(model, X_train, y_train, model_name="model"):
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.long)

    n = len(X_t)
    n_val = max(1, int(0.10 * n))
    n_tr = n - n_val
    train_ds, val_ds = random_split(
        TensorDataset(X_t, y_t),
        [n_tr, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(train_ds, batch_size=META_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=META_BATCH_SIZE, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    best_state = None
    best_val = float("inf")
    bad_epochs = 0

    model = model.to(DEVICE)

    for epoch in range(META_EPOCHS):
        model.train()
        tr_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            tr_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                vloss = criterion(logits, yb)
                val_losses.append(vloss.item())

        mean_tr = float(np.mean(tr_losses)) if tr_losses else float("nan")
        mean_val = float(np.mean(val_losses)) if val_losses else float("nan")
        print(f"{model_name} epoch {epoch+1:02d}/{META_EPOCHS} | train_loss={mean_tr:.4f} val_loss={mean_val:.4f}")

        if mean_val < best_val - 1e-5:
            best_val = mean_val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping for {model_name} at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def train_meta_learners(X_train: np.ndarray, y_train: np.ndarray):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train).astype(np.float32)

    mlp = StackingMLP(in_dim=16, n_classes=NUM_LABELS)
    mlp = _train_torch_model(mlp, Xs, y_train, model_name="MLP")

    # LinearSVC is much faster than RBF SVC for large train-split meta sets.
    svm = LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=SEED,
        max_iter=20000,
    )
    svm.fit(Xs, y_train)
    print("LinearSVC fitted.")

    bilstm = StackingBiLSTM(hidden_size=64, n_classes=NUM_LABELS)
    bilstm = _train_torch_model(bilstm, Xs, y_train, model_name="BiLSTM")

    with torch.no_grad():
        mlp_pred = mlp(torch.tensor(Xs, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()
        bilstm_pred = bilstm(torch.tensor(Xs, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    print("Train acc | MLP:", round(float((mlp_pred == y_train).mean()), 4))
    print("Train acc | LinearSVC:", round(float((svm.predict(Xs) == y_train).mean()), 4))
    print("Train acc | BiLSTM:", round(float((bilstm_pred == y_train).mean()), 4))

    return {
        "scaler": scaler,
        "mlp": mlp,
        "svm": svm,
        "bilstm": bilstm,
    }


In [ ]:
def compute_metrics_dict(y_true, y_pred):
    acc = float(accuracy_score(y_true, y_pred))
    f1m = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_each = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per = {LABEL_NAMES[i]: float(f1_each[i]) for i in range(NUM_LABELS)}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return {"accuracy": acc, "f1_macro": f1m, "f1_per_class": f1_per, "cm": cm}


def plot_confusion(cm, title):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def append_row(rows, split_key, method, y_true, y_pred, row_kind="computed"):
    m = compute_metrics_dict(y_true, y_pred)
    rows.append(
        {
            "split": split_key,
            "method": method,
            "row_kind": row_kind,
            "accuracy": m["accuracy"],
            "f1_macro": m["f1_macro"],
            "f1_entailment": m["f1_per_class"]["entailment"],
            "f1_neutral": m["f1_per_class"]["neutral"],
            "f1_contradiction": m["f1_per_class"]["contradiction"],
        }
    )


def append_reference_row(rows, split_key):
    ref_acc = 0.8408 if split_key == "trglue_mnli::test_matched" else float("nan")
    rows.append(
        {
            "split": split_key,
            "method": "REF_published_static_weighted_84.08pct",
            "row_kind": "reference_published",
            "accuracy": ref_acc,
            "f1_macro": float("nan"),
            "f1_entailment": float("nan"),
            "f1_neutral": float("nan"),
            "f1_contradiction": float("nan"),
        }
    )


In [ ]:
datasets = {}
for cfg in CONFIGS:
    print(f"Loading {DATASET_NAME} :: {cfg}")
    datasets[cfg] = load_dataset(DATASET_NAME, cfg)
    print("  splits:", list(datasets[cfg].keys()))

rows = []
trained_meta = {}
train_feature_bank = {}

for cfg in CONFIGS:
    print("\n" + "=" * 100)
    print(f"Config: {cfg}")

    # Train-only meta training set (strict no leakage).
    ds_train = datasets[cfg]["train"]
    y_train = np.array(ds_train["label"], dtype=np.int64)

    train_probs = ensure_cached_split_probs(cfg, "train", ds_train)
    X_train = build_X16(train_probs)

    print(f"Train features shape for {cfg}: {X_train.shape}")
    meta = train_meta_learners(X_train, y_train)

    trained_meta[cfg] = meta
    train_feature_bank[cfg] = {
        "X_train": X_train,
        "y_train": y_train,
        "probs": train_probs,
    }

    # Held-out evaluation.
    for sp in EVAL_SPLITS[cfg]:
        ds_eval = datasets[cfg][sp]
        y_true = np.array(ds_eval["label"], dtype=np.int64)
        split_key = f"{cfg}::{sp}"

        eval_probs = ensure_cached_split_probs(cfg, sp, ds_eval)
        X_eval = build_X16(eval_probs)
        X_eval_s = meta["scaler"].transform(X_eval).astype(np.float32)

        with torch.no_grad():
            pred_mlp = meta["mlp"](torch.tensor(X_eval_s, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()
            pred_bilstm = meta["bilstm"](
                torch.tensor(X_eval_s, dtype=torch.float32).to(DEVICE)
            ).argmax(-1).cpu().numpy()

        pred_svm = meta["svm"].predict(X_eval_s)

        pred_bert = eval_probs["bert"].argmax(axis=1)
        pred_mdeb = eval_probs["mdeberta"].argmax(axis=1)
        pred_gemma = eval_probs["gemma"].argmax(axis=1)
        pred_qwen = eval_probs["qwen"].argmax(axis=1)

        pred_majority = majority_vote_pred(
            eval_probs["bert"], eval_probs["mdeberta"], eval_probs["gemma"], eval_probs["qwen"]
        )
        pred_weighted = weighted_static_pred(
            eval_probs["bert"], eval_probs["mdeberta"], eval_probs["gemma"], eval_probs["qwen"]
        )
        pred_route = class_conditional_routing_pred(
            eval_probs["bert"], eval_probs["mdeberta"], eval_probs["gemma"], eval_probs["qwen"]
        )

        append_row(rows, split_key, "Meta_MLP_train_only_16feat", y_true, pred_mlp)
        append_row(rows, split_key, "Meta_LinearSVC_train_only_16feat", y_true, pred_svm)
        append_row(rows, split_key, "Meta_BiLSTM_train_only_16feat", y_true, pred_bilstm)

        append_row(rows, split_key, "BERT", y_true, pred_bert)
        append_row(rows, split_key, "mDeBERTa", y_true, pred_mdeb)
        append_row(rows, split_key, "Gemma", y_true, pred_gemma)
        append_row(rows, split_key, "Qwen", y_true, pred_qwen)

        append_row(rows, split_key, "Majority_vote", y_true, pred_majority)
        append_row(rows, split_key, "Weighted_static_hand_weights", y_true, pred_weighted)
        append_row(rows, split_key, "Class_conditional_routing_computed", y_true, pred_route)

        append_reference_row(rows, split_key)

        mlp_cm = compute_metrics_dict(y_true, pred_mlp)["cm"]
        bilstm_cm = compute_metrics_dict(y_true, pred_bilstm)["cm"]
        plot_confusion(mlp_cm, f"MLP (16feat) CM — {split_key}")
        plot_confusion(bilstm_cm, f"BiLSTM (16feat) CM — {split_key}")

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values(["split", "row_kind", "accuracy"], ascending=[True, True, False], na_position="last")

out_csv = ARTIFACT_DIR / "stacking_results_train_split_meta_learners_16feat_entropy.csv"
results_df.to_csv(out_csv, index=False)
print("Saved results:", out_csv)

display(results_df)


In [ ]:
metric_cols = [
    "accuracy",
    "f1_macro",
    "f1_entailment",
    "f1_neutral",
    "f1_contradiction",
]

pivot_df = results_df.pivot_table(
    index=["split", "method", "row_kind"],
    values=metric_cols,
    aggfunc="first",
).reset_index()

styled = (
    pivot_df.style
    .format({m: "{:.4f}" for m in metric_cols})
    .background_gradient(subset=["accuracy", "f1_macro"], cmap="YlGn")
    .set_properties(**{"text-align": "left"})
    .set_caption("Train-split-only stacking (16 features = 12 probs + 4 entropy)")
)

display(styled)


In [ ]:
def _normalize_importance(vals, names):
    vals = np.array(vals, dtype=np.float64)
    s = vals.sum()
    if s <= 0:
        s = 1.0
    return pd.DataFrame({"feature": names, "importance": vals / s}).sort_values("importance", ascending=False)


def _permutation_importance_custom_blocks(Xs, y, predict_fn, block_defs, n_repeats=3, seed=SEED):
    rng = np.random.default_rng(seed)
    base_pred = predict_fn(Xs)
    base_acc = accuracy_score(y, base_pred)

    names, imps = [], []
    for name, cols in block_defs:
        names.append(name)
        drops = []
        for _ in range(n_repeats):
            Xp = Xs.copy()
            perm = rng.permutation(len(Xp))
            Xp[:, cols] = Xp[perm][:, cols]
            p = predict_fn(Xp)
            drops.append(base_acc - accuracy_score(y, p))
        imps.append(float(np.mean(drops)))

    return pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False)


# Exact layout from build_X16:
# [0:3] bert probs, [3:6] mdeberta probs, [6:9] gemma probs, [9:12] qwen probs,
# [12] entropy_bert, [13] entropy_mdeberta, [14] entropy_gemma, [15] entropy_qwen.
block_defs = [
    ("bert_block", [0, 1, 2, 12]),
    ("mdeberta_block", [3, 4, 5, 13]),
    ("gemma_block", [6, 7, 8, 14]),
    ("qwen_block", [9, 10, 11, 15]),
]

for cfg in CONFIGS:
    print("\n" + "#" * 100)
    print(f"Feature importance analysis for: {cfg}")

    meta = trained_meta[cfg]
    X_train = train_feature_bank[cfg]["X_train"]
    y_train = train_feature_bank[cfg]["y_train"]
    Xs = meta["scaler"].transform(X_train).astype(np.float32)

    # MLP first-layer per-feature norms.
    mlp_w = meta["mlp"].net[0].weight.detach().cpu().numpy()  # (64, 16)
    mlp_feat_scores = np.linalg.norm(mlp_w, axis=0)
    print("\nMLP first-layer feature norms (normalized):")
    display(_normalize_importance(mlp_feat_scores, FEATURE_NAMES))

    def predict_bilstm(X_in):
        with torch.no_grad():
            return meta["bilstm"](torch.tensor(X_in, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    def predict_mlp(X_in):
        with torch.no_grad():
            return meta["mlp"](torch.tensor(X_in, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    print("MLP permutation importance (model blocks: probs+entropy together):")
    display(_permutation_importance_custom_blocks(Xs, y_train, predict_mlp, block_defs))

    print("BiLSTM permutation importance (model blocks: probs+entropy together):")
    display(_permutation_importance_custom_blocks(Xs, y_train, predict_bilstm, block_defs))

    # LinearSVC summary: no support vectors; inspect learned coefficients instead.
    svm_coef = meta["svm"].coef_  # shape: (n_classes, 16)
    svm_feat_scores = np.linalg.norm(svm_coef, axis=0)
    print("LinearSVC coefficient-norm feature importance (normalized):")
    display(_normalize_importance(svm_feat_scores, FEATURE_NAMES))


## Conclusion

This notebook keeps the strict leakage-safe protocol and adds explicit uncertainty modeling.

- Meta-learners are trained only on official `train` rows per config.
- Held-out splits are used only for evaluation.
- Feature space includes model confidence shape (`soft probs`) and model uncertainty magnitude (`entropy`).
- The sequence model is now a properly grouped **BiLSTM** over model-level timesteps (not a flat pseudo-sequence).
- The linear baseline is now **LinearSVC**, which is much more practical for 100k+ meta-train rows.

Final comparisons should be made on held-out metrics only, since this setup avoids validation-in-training leakage.
